In [2]:
!pip install -q google-genai chromadb langchain langchain-community pypdf pdf2image


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [3]:
!pip install -q python-dotenv


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [4]:
import os
from dotenv import load_dotenv
from google import genai

load_dotenv(dotenv_path="env")
API_KEY = os.getenv("GOOGLE_API_KEY")

client = genai.Client(api_key=API_KEY)
response = client.models.generate_content(
    model="gemini-3.6-flash",
    contents="hi")
print(f" {response.text.strip()}")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


 Hello! How can I help you today?


# read book

In [1]:
import os
import time
from dotenv import load_dotenv
from google import genai
from google.genai import types
from IPython.display import Markdown, display
from pypdf import PdfReader, PdfWriter

load_dotenv(dotenv_path="env")
API_KEY = os.getenv("GOOGLE_API_KEY")
client = genai.Client(api_key=API_KEY)

input_pdf_path = "Math364.pdf"
output_md_path = "Math364_Ch2.md"  
temp_page_path = "temp_page.pdf"

reader = PdfReader(input_pdf_path)

start_page = 49
end_page = 119

ocr_prompt = r"""
أنت خبير رقمنة مناهج الرياضيات العربية (OCR الرياضي المتخصص).

المهمة: استخراج محتوى الصفحة حرفياً وبدقة 100% كما هي في الكتاب دون أي تلخيص أو تصرف، مع معالجة اتجاه النصوص والمعادلات.

القواعد الإلزامية للتنسيق:
1. اتجاه الفقرات:
   - ضع كل فقرة نصية أو سطر يحتوي على كلام عربي داخل وسم HTML التالي لضبط المحاذاة:
     <div dir="rtl" style="text-align: right; font-size: 16px;">
     نص الفقرة هنا مع المعادلات المضمنة مثل $f(x) = \sin(x)$
     </div>

2. الرموز والمعادلات الرياضية:
   - استخدم $...$ للمعادلات السطرية القصيرة المدمجة مع الكلام.
   - استخدم $$...$$ لكل خطوة حل أو معادلة رئيسية في سطر مستقل لتوسيطها رياضياً.
   - عند كتابة تعليق عربي داخل سطر المعادلة الرياضية المستقلة، استخدم داخل كود اللاتكس: \text{شرح الخطوة}.

3. الهيكل والجداول:
   - احتفظ بعناوين الأمثلة، التمارين، والملاحظات الهامشية بنفس تنسيقها وترتيبها البصري في الصفحة.
   - اكتب في أعلى المخرجات: ### صفحة {page_num}
"""

config = types.GenerateContentConfig(
    temperature=0.0,
)

with open(output_md_path, "a", encoding="utf-8") as out_file:
    for page_num in range(start_page, end_page + 1):
        idx = page_num - 2 
        print(f"جاري معالجة الصفحة {page_num} من {end_page}...")

        writer = PdfWriter()
        writer.add_page(reader.pages[idx])
        with open(temp_page_path, "wb") as f:
            writer.write(f)

        try:
            uploaded_file = client.files.upload(file=temp_page_path)

            current_prompt = ocr_prompt.replace("{page_num}", str(page_num))
            response = client.models.generate_content(
                model="gemini-3.6-flash",
                contents=[uploaded_file, current_prompt],
                config=config,
            )

            page_content = f"\n\n<!-- PAGE_START_{page_num} -->\n"
            page_content += response.text.strip()
            page_content += f"\n<!-- PAGE_END_{page_num} -->\n"

            out_file.write(page_content)
            out_file.flush()

            client.files.delete(name=uploaded_file.name)

        except Exception as e:
            print(f"حدث خطأ في الصفحة {page_num}: {e}")
            time.sleep(10)
            continue

        time.sleep(2)

if os.path.exists(temp_page_path):
    os.remove(temp_page_path)

print( f"تمت معالجة الصفحات من {start_page} إلى {end_page} بنجاح وحفظها في: {output_md_path}")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


جاري معالجة الصفحة 49 من 119...
جاري معالجة الصفحة 50 من 119...
جاري معالجة الصفحة 51 من 119...
جاري معالجة الصفحة 52 من 119...
جاري معالجة الصفحة 53 من 119...
جاري معالجة الصفحة 54 من 119...
جاري معالجة الصفحة 55 من 119...
جاري معالجة الصفحة 56 من 119...
جاري معالجة الصفحة 57 من 119...
حدث خطأ في الصفحة 57: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
جاري معالجة الصفحة 58 من 119...
حدث خطأ في الصفحة 58: 503 UNAVAILABLE. {'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}
جاري معالجة الصفحة 59 من 119...
جاري معالجة الصفحة 60 من 119...
جاري معالجة الصفحة 61 من 119...
جاري معالجة الصفحة 62 من 119...
جاري معالجة الصفحة 63 من 119...
جاري معالجة الصفحة 64 من 119...
جاري معالجة الصفحة 65 من 119...
جاري معالجة الصفحة 66 

In [ ]:
from IPython.display import display, Markdown
with open("Math366_Ch4.md", "r", encoding="utf-8") as f:
    content = f.read()
display(Markdown(content))

# Document Chunking & Metadata Tagging

In [5]:
import os
import re
from langchain_core.documents import Document

file_paths = [
    "Math366_pages_12_to_40.md",  # الفصل الأول
    "Math366_Ch2.md",  # الفصل الثاني
    "Math366_Ch3.md",  # الفصل الثالث
    "Math366_Ch4.md",  # الفصل الرابع
]

pattern = r"<!-- PAGE_START_(\d+) -->(.*?)<!-- PAGE_END_\1 -->"
chunk_documents = []

for file_path in file_paths:
    if not os.path.exists(file_path):
        print(f"⚠️ تنبيه: الملف غير موجود: {file_path}")
        continue
    chapter_name = os.path.splitext(os.path.basename(file_path))[0]

    with open(file_path, "r", encoding="utf-8") as f:
        full_text = f.read()

    matches = re.findall(pattern, full_text, re.DOTALL)
    pages_data = [
        {"page_num": int(m[0]), "content": m[1].strip()} for m in matches
    ]

    print(
        f" تم العثور على {len(pages_data)} صفحة في الفصل: {chapter_name}"
    )

    for i in range(len(pages_data)):
        current_page = pages_data[i]

        if i < len(pages_data) - 1:
            next_page = pages_data[i + 1]
            combined_content = (
                f"--- صفحة {current_page['page_num']} ---\n{current_page['content']}\n\n"
                f"--- صفحة {next_page['page_num']} ---\n{next_page['content']}"
            )
            metadata = {
                "source": file_path,
                "chapter": chapter_name,
                "pages": f"{current_page['page_num']}-{next_page['page_num']}",
                "start_page": current_page["page_num"],
                "end_page": next_page["page_num"],
            }
        else:
            combined_content = f"--- صفحة {current_page['page_num']} ---\n{current_page['content']}"
            metadata = {
                "source": file_path,
                "chapter": chapter_name,
                "pages": f"{current_page['page_num']}",
                "start_page": current_page["page_num"],
                "end_page": current_page["page_num"],
            }

        doc = Document(page_content=combined_content, metadata=metadata)
        chunk_documents.append(doc)

print(f"\n✅ المجموع الكلي: تم تجهيز {len(chunk_documents)} مقطع متداخل لكامل الكتاب بنجاح!")

 تم العثور على 29 صفحة في الفصل: Math366_pages_12_to_40
 تم العثور على 53 صفحة في الفصل: Math366_Ch2
 تم العثور على 21 صفحة في الفصل: Math366_Ch3
 تم العثور على 31 صفحة في الفصل: Math366_Ch4

✅ المجموع الكلي: تم تجهيز 134 مقطع متداخل لكامل الكتاب بنجاح!


# Text Embeddings & Vector Database

In [6]:
import os
from dotenv import load_dotenv
from langchain_chroma import Chroma
from langchain_google_genai import GoogleGenerativeAIEmbeddings

load_dotenv(dotenv_path="env")
API_KEY = os.getenv("GOOGLE_API_KEY")

embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2",
    google_api_key=API_KEY,)

persist_directory = "./math366_vectorstore"

print(f"جاري تحويل وتضمين {len(chunk_documents)} مقطع لكامل فصول الكتاب...")

vectorstore = Chroma.from_documents(
    documents=chunk_documents,
    embedding=embeddings,
    persist_directory=persist_directory,)

print( f"🎉 تم بناء وحفظ قاعدة البيانات المتجهة لكامل الكتاب بنجاح في: {persist_directory}")

Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


جاري تحويل وتضمين 134 مقطع لكامل فصول الكتاب...
🎉 تم بناء وحفظ قاعدة البيانات المتجهة لكامل الكتاب بنجاح في: ./math366_vectorstore


# RAG

In [10]:
import os
from dotenv import load_dotenv
from IPython.display import Markdown, display
from langchain_chroma import Chroma
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.chat_history import BaseChatMessageHistory
from langchain_core.output_parsers import StrOutputParser
from langchain_core.prompts import ChatPromptTemplate, MessagesPlaceholder
from langchain_core.runnables import RunnablePassthrough
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain_google_genai import ChatGoogleGenerativeAI, GoogleGenerativeAIEmbeddings

load_dotenv(dotenv_path="env")
API_KEY = os.getenv("GOOGLE_API_KEY")

persist_directory = "./math366_vectorstore"
embeddings = GoogleGenerativeAIEmbeddings(
    model="gemini-embedding-2", google_api_key=API_KEY)

vectorstore = Chroma( persist_directory=persist_directory, embedding_function=embeddings)

retriever = vectorstore.as_retriever(
    search_type="similarity", search_kwargs={"k": 4})

llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash", google_api_key=API_KEY, temperature=0.2)


def format_docs_with_sources(docs):
    formatted = []
    for doc in docs:
        pages = doc.metadata.get("pages", "غير محدد")
        chapter = doc.metadata.get("chapter", "غير محدد")
        formatted.append(
            f"[مصدر المستند - الفصل: {chapter} | الصفحات المرجعية: {pages}]\n{doc.page_content}"
        )
    return "\n\n====================\n\n".join(formatted)


socratic_system_prompt = """
أنت معلم رياضيات ذكي وتربوي لمقرر ريض 366.
أسلوبك سقراطي تفاعلي بحت، هدفك مساعدة الطالب على استنتاج القواعد والحل بنفسه دون إعطائه الإجابة المباشرة فوراً.
مرجعك الأساسي هو المحتوى المرفق من الكتاب المدرسي.

القواعد الإلزامية للتفاعل:
1. ممنوع إعطاء الإجابة المباشرة أو القاعدة النهائية في أول رسالة:
   - إذا سأل الطالب سؤالاً مباشراً (مثل: "شنو اشتقاق دالة كذا؟" أو "كيف أحل هذه المسألة؟")، لا تعطه الجواب فوراً (مثلاً: لا تقل له مباشرة مشتقة sin هي cos).
   - بدلاً من ذلك، ذكّره بمفهوم مرتبط من الدروس السابقة أو اطرح عليه سؤالاً استكشافياً يلمّح له بالحل.
   - إذا كانت الزاوية مركبة $f(x) = \sin(g(x))$، وجّهه أولاً للاشتقاق الخارجي للدالة المثلثية، ثم سله: "ماذا نفعل بما بداخل الزاوية؟".

2. التدرج خطوة بخطوة:
   - لا تشرح كل الحالات الممكنة في رسالة واحدة.
   - ركّز على خطوة واحدة فقط، وانتظر رد الطالب قبل الانتقال للخطوة التالية.

3. التنسيق الرياضي:
   - استخدم $...$ للرموز والمعادلات السطرية المدمجة مع الشرح.
   - استخدم $$...$$ للخطوات الحسابية المستقلة لتوسيطها.

4. توثيق المرجع:
   - في نهاية كل رد، اكتب المرجع بالصيغة:
     ---
     **المرجع من الكتاب:** (الفصل: X - الصفحات: Y)

السياق المتاح من الكتاب:
{context}
"""

qa_prompt = ChatPromptTemplate.from_messages(
    [
        ("system", socratic_system_prompt),
        MessagesPlaceholder("chat_history"),
        ("human", "{input}"),
    ])

rag_chain = (
    RunnablePassthrough.assign(
        context=(lambda x: format_docs_with_sources(retriever.invoke(x["input"])))
    )
    | qa_prompt
    | llm
    | StrOutputParser())

store = {}


def get_session_history(session_id: str) -> BaseChatMessageHistory:
    if session_id not in store:
        store[session_id] = ChatMessageHistory()
    return store[session_id]


conversational_rag_chain = RunnableWithMessageHistory(
    rag_chain,
    get_session_history,
    input_messages_key="input",
    history_messages_key="chat_history",)

print("✅ تم تجهيز المعلم التفاعلي لكامل الكتاب بنجاح!")

<>:49: SyntaxWarning: invalid escape sequence '\s'
<>:49: SyntaxWarning: invalid escape sequence '\s'
C:\Users\user\AppData\Local\Temp\ipykernel_8596\3249093055.py:49: SyntaxWarning: invalid escape sequence '\s'
  - إذا كانت الزاوية مركبة $f(x) = \sin(g(x))$، وجّهه أولاً للاشتقاق الخارجي للدالة المثلثية، ثم سله: "ماذا نفعل بما بداخل الزاوية؟".
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.
Both GOOGLE_API_KEY and GEMINI_API_KEY are set. Using GOOGLE_API_KEY.


✅ تم تجهيز المعلم التفاعلي لكامل الكتاب بنجاح!


C:\Users\user\PycharmProjects\PythonProject\.venv\Lib\site-packages\IPython\core\interactiveshell.py:3748: LangChainDeprecationWarning: RunnableWithMessageHistory is deprecated. Use LangGraph's built-in persistence instead.
  exec(code_obj, self.user_global_ns, self.user_ns)


# Testing

In [2]:
from IPython.display import Markdown, display
query = "شنو اشتقاق sin "
response = conversational_rag_chain.invoke(
    {"input": query}, config={"configurable": {"session_id": "test_student"}})
display(Markdown(response))

NameError: name 'conversational_rag_chain' is not defined

# DSPy

In [3]:
pip install dspy-ai


[notice] A new release of pip is available: 25.1.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


   ---------------------------------------- 0.0/24.1 MB ? eta -:--:--
   ---------------------------------------- 0.0/24.1 MB ? eta -:--:--
   ---------------------------------------- 0.3/24.1 MB ? eta -:--:--
   ---------------------------------------- 0.3/24.1 MB ? eta -:--:--
    --------------------------------------- 0.5/24.1 MB 640.3 kB/s eta 0:00:37
   - -------------------------------------- 0.8/24.1 MB 721.4 kB/s eta 0:00:33
   - -------------------------------------- 0.8/24.1 MB 721.4 kB/s eta 0:00:33
   - -------------------------------------- 0.8/24.1 MB 721.4 kB/s eta 0:00:33
   - -------------------------------------- 0.8/24.1 MB 721.4 kB/s eta 0:00:33
   - -------------------------------------- 1.0/24.1 MB 553.2 kB/s eta 0:00:42
   - -------------------------------------- 1.0/24.1 MB 553.2 kB/s eta 0:00:42
   -- ------------------------------------- 1.3/24.1 MB 535.2 kB/s eta 0:00:43
   -- ------------------------------------- 1.3/24.1 MB 535.2 kB/s eta 0:00:43
   -- ---

In [38]:
import os
import dspy

api_key = os.getenv("GOOGLE_API_KEY")
gemini_lm = dspy.LM('gemini/gemini-3.6-flash', api_key=api_key)
dspy.settings.configure(lm=gemini_lm)

In [39]:
class MathTutorSignature(dspy.Signature):
    """مرشد تعليمي لمقرر الرياضيات يوجه الطالب خطوة بخطوة دون إعطاء الإجابة النهائية مباشرة."""
    
    book_context = dspy.InputField(desc="سياق الكتاب المدرسي المعتمد")
    student_question = dspy.InputField(desc="سؤال أو استفسار الطالب")
    
    next_step_guidance = dspy.OutputField(desc="الشرح التفاعلي والسؤال التوجيهي التالي للطالب")

In [40]:
class MathRiseTutorModule(dspy.Module):
    def __init__(self):
        super().__init__()
        self.tutor = dspy.ChainOfThought(MathTutorSignature)
        
    def forward(self, book_context, student_question):
        return self.tutor(book_context=book_context, student_question=student_question)

In [41]:
import os

chapter_files = [
    ("الفصل الأول", "Math366_pages_12_to_40.md"),
    ("الفصل الثاني", "Math366_Ch2.md"),
    ("الفصل الثالث", "Math366_Ch3.md"),
    ("الفصل الرابع", "Math366_Ch4.md"),
]

all_book_context = []
for ch_title, ch_file in chapter_files:
    if os.path.exists(ch_file):
        try:
            with open(ch_file, "r", encoding="utf-8") as f:
                all_book_context.append(f"=== سياق {ch_title} ({ch_file}) ===\n" + f.read())
        except Exception:
            pass

full_math366_context = "\n\n".join(all_book_context)

In [42]:
tutor_program = MathRiseTutorModule()
response = tutor_program(
    book_context=full_math366_context, 
    student_question="كيف أوجد مشتقة دالة الجيب؟")

print(response.next_step_guidance)

أهلاً بك! إيجاد مشتقة دالة الجيب يعتمد على شكل الزاوية:

1. **إذا كانت الدالة بسيطة:** 
   $$f(x) = \sin x \implies f'(x) = \cos x$$

2. **إذا كانت الزاوية دالة أخرى $g(x)$:** 
   نشتق الجيب بالنسبة للزاوية أولاً ثم نضرب في مشتقة الزاوية نفسها:
   $$f(x) = \sin(g(x)) \implies f'(x) = \cos(g(x)) \cdot g'(x)$$

---

**سؤال توجيهي:**
لتختبر فهمك، إذا كانت لدينا الدالة:
$$f(x) = \sin(3x^2)$$
ما هي مشتقة هذه الدالة باستعمال القاعدة أعلاه؟ حاول كتابة خطواتك! [[ ## next_step_guidance ## ]]


In [43]:
import google.generativeai as genai
import os

genai.configure(api_key=os.getenv("GOOGLE_API_KEY"))

# استعراض كافة النماذج المدعومة حالياً على حسابك
for m in genai.list_models():
    if 'generateContent' in m.supported_generation_methods:
        print(m.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.1-flash-lite-image
models/gemini-3.5-flash
models/gemini-3.5-flash-lite
models/gemini-omni-flash-preview
models/gemini-omni-1.1-flash
models/gemini-3.5-transcribe
models/gemini-3.6-flash
models/gemini-3.7-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.6-preview
models/gem